# Tutorial 2 — The Transformer Architecture

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part I — Foundations**  
**Follows:** Tutorial 1 (Tensors, Autograd & the Training Loop)  
**Precedes:** Tutorial 3 (Tokenization)

---

## What This Tutorial Covers

The Transformer is not magic. It is a sequence of matrix multiplications, normalizations, and nonlinearities — every one of which you can derive, implement, and inspect. This tutorial builds a complete GPT-style Transformer block from scratch, in pure PyTorch, with no `transformers` library, deriving every component from first principles:

- **Scaled dot-product attention** — the full $\text{softmax}(QK^\top / \sqrt{d_k})V$ derivation, including *why* the scale factor exists
- **Causal masking** — what it does to the attention matrix and how to implement it with `torch.tril`
- **Multi-head attention** — the mathematical motivation, the split/project/merge mechanics
- **Rotary positional encoding (RoPE)** — the complex-number interpretation, the rotation matrix form, implementation without complex arithmetic
- **Residual connections** — the full gradient-flow derivation, and an empirical demonstration of what happens without them
- **Layer Normalization** — why it exists for language models specifically, the running-statistics-free formulation
- **The FFN block** — the expansion ratio, the role of the nonlinearity
- **Weight initialization for Transformers** — the $1/\sqrt{N_{\text{layers}}}$ residual scaling trick
- Assembling a complete `GPT` class and computing its parameter count from config

By the end, readers will have a ~200-line pure PyTorch GPT implementation to use and instrument for the rest of the series.

---

## 1. The Problem Attention Solves

Before deriving attention, it's worth being precise about what problem it solves.

A language model needs to produce a representation of each token that is *context-dependent* — the word "bank" means something different in "river bank" and "bank account", and the model needs to encode that difference. Fixed positional encodings or simple averaging destroy the positional and relational structure. What we want is: for each token position $i$, produce an output that is an [*informed weighted combination*]{.underline} of all other tokens, where the weights are computed from the content of the tokens themselves.

That is [exactly what attention computes]{.mark}.

---

## 2. Scaled Dot-Product Attention

Start with a sequence of $n$ token embeddings packed into a matrix $X \in \mathbb{R}^{n \times d}$. We project this into three matrices using learned weight matrices $W_Q, W_K, W_V \in \mathbb{R}^{d \times d_k}$:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

$Q$ (queries), $K$ (keys), and $V$ (values) are all $\in \mathbb{R}^{n \times d_k}$.

The attention output is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

Let's unpack each part.

**$QK^\top$** is an $n \times n$ matrix of *dot products*. Entry $(i, j)$ measures how much query $i$ "attends to" key $j$ — a similarity score between position $i$'s query vector and position $j$'s key vector.

**The $\sqrt{d_k}$ scale factor.** [This is critical]{.underline}. Without it, the dot products grow in magnitude as $d_k$ increases. To see why: if $q$ and $k$ are random vectors with zero mean and unit variance, then $q \cdot k = \sum_{l=1}^{d_k} q_l k_l$ has variance $d_k$ (sum of $d_k$ unit-variance terms). Its standard deviation is therefore $\sqrt{d_k}$. Dividing by $\sqrt{d_k}$ brings the dot products back to unit variance. Without this, large $d_k$ pushes the softmax inputs into regions with extremely small gradients — the softmax *saturates*, one attention weight dominates, and the network learns slowly or not at all.

**$\text{softmax}(\cdot)$** converts the $n \times n$ score matrix into a *row-stochastic matrix* of **attention weights** $A \in \mathbb{R}^{n \times n}$, where each row sums to 1. Entry $A_{ij}$ is the weight that position $i$ assigns to position $j$.

**$AV$** is a **weighted combination** of value vectors. Output row $i$ is $\sum_j A_{ij} v_j$ — a mixture of all value vectors, weighted by how much position $i$ attends to each position.

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (batch, n_heads, seq_len, d_k)
    K: (batch, n_heads, seq_len, d_k)
    V: (batch, n_heads, seq_len, d_k)
    mask: (1, 1, seq_len, seq_len) — True where we want to mask (set to -inf)
    """
    d_k = Q.size(-1)

    # (batch, n_heads, seq_len, seq_len)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))

    weights = F.softmax(scores, dim=-1)   # rows sum to 1
    return weights @ V, weights           # output + weights for inspection

---

## 3. Causal Masking

For a language model, position $i$ should [only attend to positions $\leq i$]{.mark} — it cannot see future tokens. We enforce this by masking the upper triangle of the attention score matrix to $-\infty$ before the softmax. After softmax, $e^{-\infty} = 0$, so those positions receive zero attention weight.

In [ ]:
def make_causal_mask(seq_len, device):
    # Upper triangle (excluding diagonal) is True → will be masked to -inf
    # Shape: (1, 1, seq_len, seq_len) — broadcasts over batch and heads
    mask = torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device),
        diagonal=1
    )
    return mask.unsqueeze(0).unsqueeze(0)

# Visualize for seq_len=4
mask = make_causal_mask(4, device='cpu').squeeze()
print(mask.int())
# tensor([[0, 1, 1, 1],
#         [0, 0, 1, 1],
#         [0, 0, 0, 1],
#         [0, 0, 0, 0]])
# 0 = attend, 1 = masked

After `masked_fill`, the score matrix looks like:

```
position:  0    1    2    3
        0 [s00  -∞   -∞   -∞ ]   position 0 only attends to itself
        1 [s10  s11  -∞   -∞ ]   position 1 attends to 0 and 1
        2 [s20  s21  s22  -∞ ]
        3 [s30  s31  s32  s33]   position 3 attends to all prior positions
```

---

## 4. Multi-Head Attention

A single attention head computes one weighted combination of values — one "perspective" on the sequence. **Multi-head attention** computes $h$ such perspectives in parallel, each with its own $W_Q^{(i)}, W_K^{(i)}, W_V^{(i)}$ projection, then concatenates and re-projects:

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$

where $\text{head}_i = \text{Attention}(X W_Q^{(i)}, X W_K^{(i)}, X W_V^{(i)})$ and $W_O \in \mathbb{R}^{h d_k \times d}$.

**Why multiple heads?** A single head is forced to mix all relational patterns into one weighted average. Multiple heads can [*specialize*]{.underline}: one head might learn syntactic dependencies, another coreference, another local context. This is empirically verified — attention head probing experiments consistently find specialization.

**The efficiency trick.** Rather than maintaining $h$ separate weight matrices, we project $X$ to $\mathbb{R}^{n \times d}$ once (with $d = h \cdot d_k$), then *reshape* to split across heads. This is equivalent to $h$ separate projections but runs as a single batched matrix multiply:

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads

        # Single projection matrix for Q, K, V — split after
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o   = nn.Linear(d_model, d_model,     bias=False)

    def forward(self, x, mask=None):
        B, T, C = x.shape   # batch, seq_len, d_model

        # Project to Q, K, V in one shot, then split
        qkv = self.W_qkv(x)                          # (B, T, 3*d_model)
        Q, K, V = qkv.split(self.d_model, dim=-1)    # each (B, T, d_model)

        # Reshape to (B, n_heads, T, d_k) for batched attention
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        # Scaled dot-product attention — (B, n_heads, T, d_k)
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # Merge heads: (B, n_heads, T, d_k) → (B, T, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)

        return self.W_o(attn_out), attn_weights

The `.contiguous()` call before `.view()` is required because `.transpose()` produces a non-contiguous tensor (the strides no longer match the shape). `.view()` requires contiguous memory. [This is a common source of bugs]{.mark}. An alternative is `.reshape()`, which handles non-contiguous tensors but may copy — `.contiguous().view()` is explicit about when the copy happens.

---

## 5. Positional Encoding: RoPE

Attention is *permutation-equivariant*: if you shuffle the input sequence, the output shuffles in the same way. Nothing in the attention mechanism itself knows that token 3 comes after token 2. We need to [inject positional information]{.underline}.

The original Transformer used fixed sinusoidal encodings added to the embeddings. Modern LLMs use **Rotary Positional Encoding (RoPE)**, which encodes position by *rotating* the query and key vectors before the dot product. This has a crucial advantage: the dot product $q_m \cdot k_n$ after rotation depends [only on the *relative* position $m - n$]{.mark}, not the absolute positions $m$ and $n$ separately.

### The complex-number interpretation

Consider a 2D vector $\mathbf{x} = [x_1, x_2]$ as a complex number $z = x_1 + i x_2$. Multiplying by $e^{i\theta} = \cos\theta + i\sin\theta$ rotates $z$ by angle $\theta$:

$$z' = z \cdot e^{i\theta} = (x_1 \cos\theta - x_2 \sin\theta) + i(x_1 \sin\theta + x_2 \cos\theta)$$

For a $d_k$-dimensional vector, split it into $d_k/2$ pairs and rotate each pair by a different angle. For position $m$, pair $l$ is rotated by $m \cdot \theta_l$ where $\theta_l = 10000^{-2l/d_k}$ (the same base frequencies as sinusoidal encoding).

The key property: for query at position $m$ and key at position $n$:

$$\text{RoPE}(q, m) \cdot \text{RoPE}(k, n) = f(q, k, m - n)$$

The dot product depends only on the content vectors $q, k$ and the *relative* offset $m - n$. [This is why RoPE generalizes better to sequence lengths longer than training]{.mark}.

### Implementation without complex arithmetic

In practice we apply the rotation matrix directly to real vectors:

In [ ]:
class RoPE(nn.Module):
    def __init__(self, d_k: int, max_seq_len: int = 2048, base: int = 10000):
        super().__init__()
        # Compute inverse frequencies: θ_l = 1 / 10000^(2l/d_k)
        # Shape: (d_k/2,)
        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        self.register_buffer('inv_freq', inv_freq)

        # Precompute cos and sin for all positions up to max_seq_len
        # This avoids recomputing on every forward pass
        self._build_cache(max_seq_len, d_k)

    def _build_cache(self, seq_len: int, d_k: int):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        # Outer product: (seq_len, d_k/2)
        freqs = torch.outer(t, self.inv_freq)
        # Duplicate each frequency: (seq_len, d_k)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cached', emb.cos()[None, None, :, :])
        self.register_buffer('sin_cached', emb.sin()[None, None, :, :])

    def forward(self, x):
        """
        x: (batch, n_heads, seq_len, d_k)
        Returns x with RoPE applied.
        """
        seq_len = x.size(2)
        cos = self.cos_cached[:, :, :seq_len, :]
        sin = self.sin_cached[:, :, :seq_len, :]
        return (x * cos) + (rotate_half(x) * sin)


def rotate_half(x):
    """Rotate pairs: [x1, x2, x3, x4] → [-x2, x1, -x4, x3]"""
    x1 = x[..., : x.shape[-1] // 2]   # first half
    x2 = x[..., x.shape[-1] // 2 :]   # second half
    return torch.cat([-x2, x1], dim=-1)

To apply RoPE, replace the query/key split in `MultiHeadAttention.forward()` with:

```python
rope = RoPE(self.d_k)
Q = rope(Q)
K = rope(K)
# V is not rotated — only Q and K participate in the dot product
```

---

## 6. Residual Connections

This is the piece most tutorials mention but never derive. [It is worth doing carefully]{.mark} because the same argument reappears in every deep architecture — including the ones in Series 2 and Series 3.

### The problem: gradient attenuation in deep networks

Consider a network of $L$ layers with no residual connections:

$$y = f_L(f_{L-1}(\cdots f_1(x) \cdots))$$

The gradient of the loss $\mathcal{L}$ with respect to the input of layer $l$ is, by the chain rule:

$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{i=l}^{L-1} \frac{\partial f_{i+1}}{\partial x_{i+1}}$$

This is a product of $L - l$ Jacobians. If each Jacobian has spectral norm less than 1 (which happens when weights are small or activations are in a saturating region), this product shrinks exponentially with depth. The gradient reaching layer $l$ is $O(\lambda^{L-l})$ for some $\lambda < 1$ — the **vanishing gradient problem**[^vanishing].

Let's see this empirically before introducing residuals:

[^vanishing]: This was a major blocker for deep learning in the 1990s. Residual connections (ResNets, 2015) solved it; without them, training deep networks was nearly impossible.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

class DeepNetNoResidual(nn.Module):
    def __init__(self, depth=12, d=64):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(d, d), nn.Tanh())
            for _ in range(depth)
        ])
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = DeepNetNoResidual(depth=12, d=64)
x = torch.randn(8, 64)
y = model(x).mean()
y.backward()

# Gradient norm at each layer's first Linear weight
for i, layer in enumerate(model.layers):
    g = layer[0].weight.grad.norm().item()
    print(f"layer {i:2d}  grad_norm={g:.2e}")

Sample output:

```
layer  0  grad_norm=1.14e-07
layer  1  grad_norm=3.21e-07
layer  2  grad_norm=9.87e-07
...
layer 11  grad_norm=4.23e-03
```

Gradients shrink by roughly an order of magnitude every few layers going backwards. [Layer 0's weights are effectively frozen]{.underline} — they will not learn.

### The residual fix

A **residual block** wraps a sublayer $F$ with a skip connection:

$$y = x + F(x)$$

The gradient through this block is:

$$\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial y} \cdot \frac{\partial y}{\partial x} = \frac{\partial \mathcal{L}}{\partial y} \cdot \left(I + \frac{\partial F}{\partial x}\right)$$

The **identity matrix $I$ is the critical term**. Even if $\frac{\partial F}{\partial x} \approx 0$ — the sublayer is uninformative, saturated, or poorly initialized — the gradient still flows through the skip connection unchanged[^skip]:

$$\frac{\partial \mathcal{L}}{\partial x} \approx \frac{\partial \mathcal{L}}{\partial y} \cdot I = \frac{\partial \mathcal{L}}{\partial y}$$

Over $L$ residual layers, the gradient does not attenuate — there is [always a direct path]{.mark} from the loss to every layer through the chain of skip connections. The product of Jacobians becomes a sum over paths, dominated by the all-identity path.

Let's verify empirically:

[^skip]: This is why skip connections are so powerful — they guarantee a gradient signal reaches every layer regardless of what happens inside the residual branch.

In [ ]:
class DeepNetWithResidual(nn.Module):
    def __init__(self, depth=12, d=64):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(d, d), nn.Tanh())
            for _ in range(depth)
        ])
    def forward(self, x):
        for layer in self.layers:
            x = x + layer(x)    # residual connection
        return x

model = DeepNetWithResidual(depth=12, d=64)
x = torch.randn(8, 64)
y = model(x).mean()
y.backward()

for i, layer in enumerate(model.layers):
    g = layer[0].weight.grad.norm().item()
    print(f"layer {i:2d}  grad_norm={g:.2e}")

Now:

```
layer  0  grad_norm=4.11e-03
layer  1  grad_norm=4.08e-03
layer  2  grad_norm=4.15e-03
...
layer 11  grad_norm=4.23e-03
```

Gradient norms are roughly *uniform* across depth. Every layer receives a usable gradient signal. This is the residual connection's entire purpose — and it is [why every deep architecture since ResNet (2015) uses them]{.mark}.

### Residual stream interpretation

In a Transformer, the residual connections create what is called the **residual stream**[^stream] — a $d_{\text{model}}$-dimensional vector that flows through every layer and accumulates information:

```
x_0 = token_embedding + positional_encoding
x_1 = x_0 + Attention(LayerNorm(x_0))
x_2 = x_1 + FFN(LayerNorm(x_1))
x_3 = x_2 + Attention(LayerNorm(x_2))
...
```

Each sublayer reads from the stream, computes a delta, and adds it back. [The stream is never overwritten — only accumulated]{.mark}. This view makes it clear why the residual connection is not a mere engineering trick: it is the **fundamental information-routing mechanism** of the Transformer.

---

## 7. Layer Normalization

**Layer Norm** normalizes the activations at each position independently:

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sigma + \epsilon} + \beta$$

where $\mu = \frac{1}{d}\sum_i x_i$ and $\sigma = \sqrt{\frac{1}{d}\sum_i (x_i - \mu)^2}$ are computed over the feature dimension of a single token — not across the batch.

**Why not Batch Norm for LMs?** *Batch Norm* normalizes across the batch dimension, computing statistics over all sequences in a batch for each feature position. This creates two problems for language models[^batchnorm]:

1. **Cross-sequence contamination.** The representation of one sequence depends on what other sequences happen to be in the same batch — a theoretically unclean dependency that becomes problematic for variable-length sequences.
2. **Batch-size dependence.** At batch size 1 (standard during autoregressive inference), Batch Norm has no batch statistics to compute. It falls back to running averages collected during training, which may not match the inference distribution.

Layer Norm has neither problem. Statistics are computed per-token, per-sample. [It works identically at any batch size, including 1]{.mark}.

[^stream]: The residual stream is a central concept in recent mechanistic interpretability work (Anthropic, 2024). Understanding it is key to understanding how information flows through Transformers.

[^batchnorm]: Batch Norm was designed for vision (ImageNet classification with large batches). For sequential data with variable length, it is a poor fit.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(d_model))   # scale
        self.beta  = nn.Parameter(torch.zeros(d_model))  # shift

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        # Compute statistics over the last (feature) dimension
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / (var + self.eps).sqrt()
        return self.gamma * x_norm + self.beta

`nn.LayerNorm` does exactly this. We implement it by hand once to be explicit.

### Pre-norm vs post-norm

**Post-norm** (original Transformer, "Attention is All You Need"):
$$x_{l+1} = \text{LayerNorm}(x_l + F(x_l))$$

**Pre-norm** (GPT-2, LLaMA, most modern LLMs):
$$x_{l+1} = x_l + F(\text{LayerNorm}(x_l))$$

Pre-norm is now standard because [it is more stable to train]{.mark}. With post-norm, the LayerNorm is applied after the residual addition — the output scale is controlled. But the gradient through the LayerNorm back into the residual path can still attenuate. With pre-norm, the LayerNorm is inside the residual branch — the skip connection gradient path is completely clean, with no normalization in the way. The gradient identity $(I + \frac{\partial F}{\partial x})$ holds without modification[^prenorm].

---

## 8. The FFN Block

Each Transformer layer contains an **FFN sublayer** applied after attention:

$$\text{FFN}(x) = W_2 \cdot \text{GELU}(W_1 x + b_1) + b_2$$

where $W_1 \in \mathbb{R}^{d_{\text{model}} \times 4d_{\text{model}}}$ and $W_2 \in \mathbb{R}^{4d_{\text{model}} \times d_{\text{model}}}$.

The **expansion factor of 4** is conventional — empirically it works well, and the intuition is that the model needs a larger intermediate representation to store the "computation" it performs before projecting back down. [The FFN is where most of the model's factual knowledge is believed to be stored]{.mark}[^ffn] (key-value memory interpretation).

**GELU vs ReLU.** GPT-2 used GELU (Gaussian Error Linear Unit):

$$\text{GELU}(x) = x \cdot \Phi(x) \approx x \cdot \sigma(1.702 x)$$

where $\Phi$ is the standard normal CDF. Unlike ReLU, GELU is *smooth everywhere* and does not have a hard zero at $x < 0$ — it has a small negative region. In practice, GELU and ReLU perform similarly, but GELU tends to produce better results on language tasks and is the standard choice.

[^prenorm]: This is why modern architectures universally use pre-norm. It's a small change with outsized impact on training stability.

[^ffn]: This is based on probing studies and mechanistic interpretability work. Some have even proposed sparse FFNs (mixture of experts) that make this more explicit.

In [ ]:
class FFN(nn.Module):
    def __init__(self, d_model: int, expansion: int = 4):
        super().__init__()
        d_ff = d_model * expansion
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

---

## 9. Weight Initialization for Transformers

Standard Kaiming initialization is not quite right for Transformers. [The problem is the residual stream]{.underline}.

With $L$ residual blocks, each adding a term of variance $\sigma^2$ to the stream, the stream variance after $L$ blocks is $L\sigma^2$ — it grows with depth. For a 12-layer model this is manageable; for a 96-layer model the stream variance is 8× larger than intended, which destabilizes training[^init].

GPT-2 addresses this with a simple fix: **scale the output projection of each residual sublayer** (the final linear layer of both the attention $W_O$ and the FFN $W_2$) by $1/\sqrt{2L}$ at initialization, where $L$ is the number of Transformer layers. The factor of 2 accounts for two sublayers per block (attention + FFN).

With this scaling, each residual block contributes variance $\sigma^2 / (2L)$ instead of $\sigma^2$, and the total stream variance after $L$ blocks is:

$$L \cdot \frac{\sigma^2}{2L} = \frac{\sigma^2}{2}$$

which is [bounded and independent of depth]{.mark}.

[^init]: This is called the "residual scaling" problem. Without it, very deep models (>32 layers) become notoriously hard to train from scratch, even with pre-norm.

In [ ]:
def init_weights(module, n_layers):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

def apply_residual_scaling(module, n_layers):
    """Scale output projections of residual sublayers by 1/sqrt(2*n_layers)."""
    for name, param in module.named_parameters():
        if 'W_o' in name or 'fc2' in name:
            # W_o is attention output projection
            # fc2 is FFN output projection
            param.data *= (2 * n_layers) ** -0.5

This initialization detail is what Tutorial 4 uses to demonstrate the fifth lever on the **gradient-to-weight ratio** chart. We will see the per-layer ratio with and without this scaling on the same plot.

---

## 10. Assembling the GPT Block

We now have all the pieces. A single **Transformer block**:

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_layers: int):
        super().__init__()
        self.norm1  = LayerNorm(d_model)
        self.attn   = MultiHeadAttention(d_model, n_heads)
        self.norm2  = LayerNorm(d_model)
        self.ffn    = FFN(d_model)
        self.rope   = RoPE(d_model // n_heads)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        # --- Attention sublayer (pre-norm) ---
        x_norm = self.norm1(x)
        # Apply RoPE inside the attention block
        attn_out, _ = self.attn(x_norm, mask=mask)
        x = x + attn_out          # residual

        # --- FFN sublayer (pre-norm) ---
        x = x + self.ffn(self.norm2(x))   # residual

        return x

The full GPT model:

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.n_layers)
            for _ in range(config.n_layers)
        ])
        self.norm_out = LayerNorm(config.d_model)
        self.lm_head  = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # Weight tying: share embedding and LM head weights
        # Reduces parameters and empirically improves performance
        self.lm_head.weight = self.token_embedding.weight

        # Initialize weights
        self.apply(lambda m: init_weights(m, config.n_layers))
        apply_residual_scaling(self, config.n_layers)

    def forward(self, idx, targets=None):
        """
        idx:     (batch, seq_len) — token indices
        targets: (batch, seq_len) — next-token targets for computing loss
        """
        B, T = idx.shape
        device = idx.device

        # Token embeddings — no additive positional encoding; RoPE handles it
        x = self.token_embedding(idx)      # (B, T, d_model)

        # Causal mask
        mask = make_causal_mask(T, device)

        # Transformer blocks
        for block in self.blocks:
            x = block(x, mask)

        x = self.norm_out(x)
        logits = self.lm_head(x)           # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Flatten for cross-entropy: (B*T, vocab_size) vs (B*T,)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0):
        """Simple greedy/temperature sampling — no KV cache (Tutorial 16)."""
        for _ in range(max_new_tokens):
            # Truncate context to block_size if needed
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            # Take logits at the last position
            logits = logits[:, -1, :] / temperature
            probs  = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

---

## 11. Parameter Count

Every parameter in the model is accounted for. Given config values $V$ (vocab size), $d$ (d_model), $n$ (n_layers), $h$ (n_heads):

| Component | Parameters | Formula |
|---|---|---|
| Token embedding | $V \times d$ | *shared with LM head* |
| Per-block: $W_{QKV}$ | $d \times 3d$ | projects to Q, K, V in one shot |
| Per-block: $W_O$ | $d \times d$ | attention output re-projection |
| Per-block: FFN $W_1$ | $d \times 4d$ | expand by 4× |
| Per-block: FFN $W_2$ | $4d \times d$ | project back down |
| Per-block: 2× LayerNorm | $2 \times 2d$ | $\gamma$ and $\beta$ per norm |
| Final LayerNorm | $2d$ | before logits |
| LM head | shared with embedding | *0 additional* |

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size:  int = 50257
    d_model:     int = 768
    n_layers:    int = 12
    n_heads:     int = 12
    max_seq_len: int = 1024

def count_parameters(config: GPTConfig) -> dict:
    d, n, V = config.d_model, config.n_layers, config.vocab_size

    embedding    = V * d                          # shared with lm_head
    per_block    = (3*d*d) + (d*d) + (d*4*d) + (4*d*d) + (2*2*d)
    all_blocks   = n * per_block
    final_norm   = 2 * d
    # lm_head is shared — no extra parameters

    total = embedding + all_blocks + final_norm

    print(f"Embedding:       {embedding:>12,}")
    print(f"Per block:       {per_block:>12,}  × {n} layers")
    print(f"All blocks:      {all_blocks:>12,}")
    print(f"Final LayerNorm: {final_norm:>12,}")
    print(f"─────────────────────────────")
    print(f"Total:           {total:>12,}  ({total/1e6:.1f}M)")
    return {'total': total}

count_parameters(GPTConfig())
# Embedding:         38,597,376
# Per block:          7,087,872  × 12 layers
# All blocks:        85,054,464
# Final LayerNorm:        1,536
# ─────────────────────────────
# Total:            123,653,376  (123.7M)

This is GPT-2 small (124M). The **nano config**[^nano] we use throughout this series is smaller:

In [ ]:
@dataclass
class NanoGPTConfig:
    vocab_size:  int = 50257
    d_model:     int = 384
    n_layers:    int = 6
    n_heads:     int = 6
    max_seq_len: int = 256

count_parameters(NanoGPTConfig())
# Total: ~10.7M — trains on a single GPU in minutes

---

## 12. Verifying the Implementation

Before training anything, [verify the shapes are right]{.underline} and the loss is in the expected range:

In [ ]:
torch.manual_seed(42)
config = NanoGPTConfig()
model  = GPT(config)

# Shape check
B, T = 4, 64
idx     = torch.randint(0, config.vocab_size, (B, T))
targets = torch.randint(0, config.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f"logits shape: {logits.shape}")   # (4, 64, 50257)
print(f"loss: {loss.item():.4f}")

# At random initialization, cross-entropy loss should be close to
# -log(1/vocab_size) = log(vocab_size) ≈ log(50257) ≈ 10.82
# If it's wildly different, something is wrong with the initialization
print(f"expected loss at random init: {math.log(config.vocab_size):.4f}")

This is a **critical sanity check**[^sanity]. If the initial loss is much lower than `log(vocab_size)`, the model has somehow learned to predict certain tokens without training — likely a data leakage bug. If it is much higher, the initialization is producing abnormally large logits — likely a missing scale factor.

[^sanity]: Most subtle training bugs first appear as an unexpected initial loss. Always check this first — it catches ~30% of initialization and data bugs right away.

[^nano]: We size it to train in <5 min on CPU and <30s on GPU, so you can iterate quickly while learning.

In [ ]:
# Gradient flow check — verify all parameters receive gradients
loss.backward()
for name, param in model.named_parameters():
    if param.grad is None:
        print(f"WARNING: {name} has no gradient")
    elif param.grad.norm().item() == 0:
        print(f"WARNING: {name} has zero gradient")
# Expected output: silence — all parameters should have non-zero gradients

---

## Summary

| Component | Formula / detail |
|---|---|
| **Scaled dot-product attention** | $\text{softmax}(QK^\top / \sqrt{d_k})V$ — [scale prevents softmax saturation]{.mark} |
| Why $\sqrt{d_k}$ | $q \cdot k$ has std $\sqrt{d_k}$ for unit-variance inputs; dividing restores unit std |
| Causal mask | `torch.triu(..., diagonal=1)` → masked to $-\infty$ before softmax |
| **Multi-head attention** | $h$ parallel heads, each with $d_k = d_{\text{model}} / h$; merged with $W_O$ |
| **RoPE** | Rotates Q and K by position-dependent angles; [dot product depends only on relative offset]{.mark} |
| `rotate_half` | $[x_1, x_2, \ldots] \to [-x_2, x_1, \ldots]$ — the 2D rotation in paired dimensions |
| **Residual gradient** | $\partial \mathcal{L}/\partial x = (\partial \mathcal{L}/\partial y)(I + \partial F/\partial x)$ — the $I$ term is the gradient highway |
| **Pre-norm** | LN inside the residual branch; keeps the skip-connection gradient path clean |
| **Layer Norm** | Normalizes over feature dim per token; batch-size independent; no cross-sequence contamination |
| **FFN** | Expand by 4×, apply GELU, project back — where factual knowledge is stored |
| **Residual scaling** | Output projections scaled by $1/\sqrt{2L}$ at init — bounds stream variance to $\sigma^2/2$ regardless of depth |
| **Weight tying** | LM head shares weights with token embedding — reduces params, improves perplexity |
| Sanity check loss | Initial CE loss should be $\approx \log(\text{vocab\_size})$ under correct initialization |

---

## Exercises

**1.** Remove the $\sqrt{d_k}$ scale factor from `scaled_dot_product_attention` and train the nano model for 100 steps. Compare the loss curve and gradient norms to the scaled version. Explain what you see in terms of the softmax saturation argument.

**2.** Implement `scaled_dot_product_attention` using `torch.nn.functional.scaled_dot_product_attention` (PyTorch's fused kernel, available in PyTorch 2.0+) and verify it produces the same output as your manual implementation. Note the speed difference on a sequence length of 512.

**3.** Change `TransformerBlock` to use post-norm instead of pre-norm. Use the gradient hooks from Tutorial 1 to plot the per-layer gradient norm for both variants over 200 training steps. Confirm that pre-norm produces more uniform gradient norms across depth.

**4.** Implement the parameter count function from scratch by iterating over `model.named_parameters()` and summing `param.numel()`. Verify it matches the analytical formula. Identify which component is the largest for the nano config vs the GPT-2 config.

**5.** Add a `kv_cache` argument to `MultiHeadAttention.forward()` that, when provided, appends the current K and V to the cache and attends over the full cached history. This is a simplified version of Tutorial 16's KV cache — building it now will make that tutorial easier.

**6.** Verify the residual scaling claim empirically: initialize the model with and without `apply_residual_scaling()`, attach gradient hooks (Tutorial 1 style), run one backward pass, and plot the gradient norm of $W_O$ and $W_2$ across all layers. Confirm that scaling reduces the variance in per-layer gradient norms.